# R2-06 · Clasificación y clustering — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R2 · *Científico de Datos* · Semana 12 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Entrenar un **clasificador** y desconfiar de la exactitud con clases desbalanceadas.
2. Leer la **matriz de confusión** y distinguir los dos tipos de error.
3. Usar **precisión y recall** para decidir qué error es más caro.
4. Agrupar con **K-Means** (con escalado) e **interpretar** los clusters.

**Competencia de salida:** entrenar y evaluar un clasificador con clases desbalanceadas, y aplicar clustering con K-Means distinguiendo cuándo un problema es supervisado y cuándo no.

**Dato real:** PM2.5 diario 2020–2025 de 14 estaciones SINCA (Ministerio del Medio Ambiente).

**Entregable:** que las **6 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# "En vivo o caché": usa los archivos locales; si faltan (ej. Colab), se bajan del repo.
for archivo in ["aire_diario.csv", "estaciones.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)

df = pd.read_csv("aire_diario.csv", parse_dates=["fecha"])   # PM2.5 diario por estación
estaciones = pd.read_csv("estaciones.csv")                    # comuna y región de cada estación
print("mediciones:", df.shape, "| estaciones:", estaciones.shape)
df.head(3)

## Dos preguntas, dos tipos de aprendizaje

- **Supervisado (clasificación):** *«Tengo seis años de días ya etiquetados: sé cuáles superaron
  la norma de PM2.5. ¿Puede un modelo aprender a anticipar si **mañana** la superará?»*
  Hay una respuesta correcta (`y`) con la que aprender.
- **No supervisado (clustering):** *«Tengo 14 estaciones con su contaminación mes a mes, pero
  nadie las ha clasificado. ¿Se agrupan solas en **tipos de ciudad** según cuándo se ensucia
  su aire?»* No hay respuesta correcta: el algoritmo busca parecidos.

> 📏 **La norma.** La norma primaria de calidad del aire para PM2.5 en 24 horas es
> **50 µg/m³** (D.S. 12/2011 del Ministerio del Medio Ambiente). Un día «sobre la norma» es
> un día en que la población respiró aire que la ley considera dañino para la salud.

### Preparación

In [ ]:
# Preparación: ordenamos por estación y fecha y construimos las variables.
df = df.sort_values(["estacion", "fecha"])
# lo que SÍ sabemos la víspera: el PM2.5 de ayer en la misma estación
df["pm25_ayer"] = df.groupby("estacion")["pm25"].shift(1)
df["mes"] = df["fecha"].dt.month
df["invierno"] = df["mes"].isin([5, 6, 7, 8]).astype(int)   # mayo a agosto
# solo días con el dato de hoy y el de ayer
d = df.dropna(subset=["pm25", "pm25_ayer"]).copy()
d["sobre_norma"] = (d["pm25"] > 50).astype(int)              # 1 = día sobre la norma
print("días utilizables:", len(d), "| días sobre la norma:", int(d["sobre_norma"].sum()))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## 1. Clasificación: predecir una categoría

Es el mismo flujo de R2-04 y R2-05 —separar prueba, `fit`, `predict`— con un cambio: el objetivo `y` ahora es una **categoría** (0 = día bajo la norma, 1 = día sobre la norma).

Antes de modelar, mira **cuántos** positivos hay. Si son pocos, las clases están **desbalanceadas** y conviene dividir con `stratify=y`, para que entrenamiento y prueba tengan la **misma proporción** de días sobre la norma.

> 🧠 **Analogía.** No estimas *cuántos* µg/m³ habrá (eso sería regresión): decides si el día cae en la pila «normal» o en la pila «sobre la norma».

### ✍️ Ejercicio 1 — Prepara X, y y una división estratificada

In [ ]:
# TODO: X con las columnas "pm25_ayer" e "invierno"; y con "sobre_norma"
X = ...
y = ...
# TODO: proporción de días sobre la norma (el promedio de una columna 0/1)
tasa_positivos = ...
# TODO: divide con test_size=0.3, random_state=42 y stratify=y
X_train, X_test, y_train, y_test = ...
print("tasa de positivos:", tasa_positivos)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert list(X_train.columns) == ["pm25_ayer", "invierno"], "X debe tener pm25_ayer e invierno, en ese orden"
    assert 0.02 < tasa_positivos < 0.08, "La tasa de días sobre la norma debería rondar el 4 %"
    assert abs(len(X_test) - 0.3 * len(X)) <= 1, "Usa test_size=0.3"
    assert abs(y_train.mean() - y_test.mean()) < 0.002, "Falta stratify=y: las proporciones no calzan"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "X = d[['pm25_ayer', 'invierno']], y = d['sobre_norma'], tasa = y.mean() y train_test_split(X, y, test_size=0.3, random_state=42, stratify=y).")
    print("   Detalle:", e)

## 2. La trampa de la exactitud

La **exactitud** (*accuracy*) es el % de días bien clasificados. Suena bien, pero con clases desbalanceadas **engaña**: un «modelo» que **siempre** dice «hoy no se supera la norma» acierta en todos los días normales… y nunca avisa de un día malo. Entrena una **regresión logística** (el clasificador lineal clásico) y compárala con ese modelo tonto.

### ✍️ Ejercicio 2 — Entrena el clasificador y compáralo con el modelo que siempre dice «no»

In [ ]:
modelo = LogisticRegression(max_iter=1000)
# TODO: entrena con (X_train, y_train) y predice sobre X_test
...
y_pred = ...
# TODO: exactitud del modelo y la del modelo que siempre predice 0
exactitud = ...
exactitud_tonta = ...
print("modelo:", exactitud, "| siempre no:", exactitud_tonta)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    ref = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    assert abs(float(exactitud) - accuracy_score(y_test, ref.predict(X_test))) < 0.005
    assert abs(float(exactitud_tonta) - (y_test == 0).mean()) < 1e-9
    assert exactitud_tonta > 0.9, "El modelo tonto también supera el 90 %: ahí está la trampa"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'modelo.fit(X_train, y_train); y_pred = modelo.predict(X_test); accuracy_score(y_test, y_pred) y (y_test == 0).mean() para el modelo que siempre dice 0.')
    print("   Detalle:", e)

## 3. La matriz de confusión: ¿en qué se equivoca?

La exactitud resume todo en un número; la **matriz de confusión** separa los aciertos y los **dos tipos de error**:

| | predice «normal» | predice «sobre la norma» |
|---|---|---|
| **real: normal** | verdadero negativo (TN) | **falsa alarma** (FP) |
| **real: sobre la norma** | **día malo que se escapó** (FN) | verdadero positivo (TP) |

Con `confusion_matrix(y_test, y_pred).ravel()` obtienes los cuatro números en el orden `tn, fp, fn, tp`.

### ✍️ Ejercicio 3 — Cuenta los aciertos y los dos tipos de error

In [ ]:
# TODO: matriz de confusión entre lo real (y_test) y lo predicho (y_pred)
cm = ...
# TODO: desarma la matriz en sus 4 números (en este orden: tn, fp, fn, tp)
tn, fp, fn, tp = ...
print(cm)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert tn + fp + fn + tp == len(y_test)
    assert tp + fn == int(y_test.sum()), "tp + fn deben sumar todos los días realmente sobre la norma"
    assert fn > 0, "Con este modelo se escapan días malos (fn > 0)"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'cm = confusion_matrix(y_test, y_pred) y tn, fp, fn, tp = cm.ravel().')
    print("   Detalle:", e)

In [ ]:
# (ilustración) La matriz de confusión dibujada
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["normal", "sobre la norma"], cmap="Blues")
plt.title("¿Supera mañana la norma? — conjunto de prueba")
plt.show()

## 4. Precisión, recall y el error más caro

Dos métricas miran **solo** a la clase que importa (el día sobre la norma):

- **Precisión** = de los días que el modelo **alertó**, ¿cuántos eran realmente malos? (pocas falsas alarmas)
- **Recall** (sensibilidad) = de los días **realmente** malos, ¿cuántos detectó? (pocos días que se escapan)

Para la **autoridad sanitaria** los errores no cuestan lo mismo. Un día malo que se escapa (FN) es población —niñas, niños, personas mayores, enfermos respiratorios— expuesta **sin aviso ni medidas**. Una falsa alarma (FP) es una restricción innecesaria (calefacción a leña, actividad física escolar) con costo económico y de credibilidad. Si el FN es más caro, conviene **subir el recall** aunque baje la precisión.

Una forma de lograrlo: `class_weight="balanced"` le dice al modelo que equivocarse con la clase rara pesa más.

### ✍️ Ejercicio 4 — Compara precisión y recall del modelo normal y del modelo balanceado

In [ ]:
# TODO: precisión y recall del modelo normal (y_test vs y_pred)
precision_base = ...
recall_base = ...

# TODO: entrena el mismo modelo con class_weight="balanced" y predice X_test
modelo_bal = ...
y_pred_bal = ...
precision_bal = ...
recall_bal = ...
print("normal:", precision_base, recall_base, "| balanceado:", precision_bal, recall_bal)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert abs(float(precision_base) - precision_score(y_test, y_pred)) < 1e-9
    assert abs(float(recall_base) - recall_score(y_test, y_pred)) < 1e-9
    assert recall_bal > recall_base + 0.2, "El modelo balanceado debería detectar muchos más días malos"
    assert precision_bal < precision_base, "…a cambio de más falsas alarmas (menor precisión)"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "precision_score / recall_score sobre (y_test, y_pred); luego LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train, y_train).")
    print("   Detalle:", e)

In [ ]:
# (ilustración) El intercambio, en días concretos del conjunto de prueba
for nombre, pred in [("normal", y_pred), ("balanceado", y_pred_bal)]:
    _, fp_, fn_, _ = confusion_matrix(y_test, pred).ravel()
    print(f"{nombre:>10}: {fn_:>4} días malos se escapan | {fp_:>4} falsas alarmas")

## 5. Clustering: descubrir grupos sin etiquetas

Cambiamos de mundo. Ahora **no** hay una respuesta correcta. Resumimos cada estación en su **perfil**: el PM2.5 promedio de cada mes (una fila por estación, 12 columnas). Le pedimos a **K-Means** que forme **3 grupos** de estaciones parecidas.

K-Means mide **distancias**, así que primero **estandarizamos** con `StandardScaler` (cada mes queda con media 0 y desviación 1): así ningún mes domina solo por tener números más grandes.

> 🧠 **Analogía.** Te pasan 14 fichas de ciudades sin clasificar y las separas en pilas según cómo se ve su año de contaminación. Nadie te dijo qué pilas hacer.

### ✍️ Ejercicio 5 — Arma el perfil mensual y agrupa las estaciones con K-Means

In [ ]:
# TODO: tabla estación × mes con el PM2.5 promedio (pivot_table con aggfunc="mean")
perfil = ...
# TODO: estandariza el perfil con StandardScaler().fit_transform(...)
Z = ...
km = KMeans(n_clusters=3, random_state=42, n_init=10)
# TODO: etiqueta de cluster de cada estación (km.fit_predict(Z)), como Serie con índice = estación
grupos = ...
print(grupos)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert perfil.shape == (14, 12), "El perfil debe tener 14 estaciones (filas) y 12 meses (columnas)"
    assert abs(float(np.mean(Z))) < 1e-6, "Z debe estar estandarizado (media 0)"
    assert len(grupos) == 14 and grupos.nunique() == 3
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "perfil = df.pivot_table(index='estacion', columns='mes', values='pm25', aggfunc='mean'); Z = StandardScaler().fit_transform(perfil); grupos = pd.Series(km.fit_predict(Z), index=perfil.index).")
    print("   Detalle:", e)

In [ ]:
# (ilustración) Cada línea es una estación, coloreada por su cluster
colores = ["#4240e5", "#ff6b35", "#2a9d8f"]
fig, ax = plt.subplots(figsize=(8, 4.5))
for est, fila in perfil.iterrows():
    ax.plot(fila.index, fila.values, color=colores[grupos[est]], alpha=0.8)
ax.axhline(50, ls="--", color="gray", lw=1)
ax.set_xticks(range(1, 13))
ax.set_xlabel("Mes"); ax.set_ylabel("PM2.5 promedio (µg/m³)")
ax.set_title("Perfil mensual de PM2.5 por estación, coloreado por cluster")
plt.tight_layout(); plt.show()

## 6. Interpretar los grupos

Un cluster no trae nombre: **tú** se lo pones mirando su perfil promedio y quiénes lo integran. Calcula el perfil promedio de cada grupo, identifica el grupo con el **invierno más severo** (mayor PM2.5 en junio) y lista sus estaciones. Luego cruza con `estaciones` para ver dónde están.

### ✍️ Ejercicio 6 — Ponle nombre al grupo del invierno más severo

In [ ]:
# TODO: perfil promedio de cada grupo (agrupa perfil por grupos y promedia)
perfil_grupo = ...
# TODO: el cluster con mayor PM2.5 en junio (columna 6) — usa idxmax()
cluster_invierno = ...
# TODO: lista de estaciones que pertenecen a ese cluster
estaciones_invierno = ...
print("Grupo de invierno más severo:", estaciones_invierno)

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    assert perfil_grupo.shape == (3, 12)
    assert "Las Encinas Temuco" in estaciones_invierno and "Nielol" in estaciones_invierno
    assert "Antofagasta" not in estaciones_invierno
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'perfil.groupby(grupos).mean(); perfil_grupo[6].idxmax(); list(grupos[grupos == cluster_invierno].index).')
    print("   Detalle:", e)

## ✍️ Reflexión (no se corrige, pero es lo más importante)

1. Con el modelo balanceado se escapan muchos menos días malos, pero suben las falsas alarmas.
   Si fueras la **SEREMI de Salud**, ¿con cuál te quedarías y cómo se lo explicarías a la
   ciudadanía que tuvo que apagar su estufa un día que al final no superó la norma?
2. El clustering separó a Temuco del resto, sin que nadie le dijera nada de leña ni de latitud.
   ¿Qué **política** distinta pediría cada grupo? ¿Qué le falta al algoritmo para saber *por qué*
   se parecen?

> Un modelo te da números; la decisión sobre **qué error tolerar** es pública y se declara.

## Errores típicos

- **Celebrar un 96 % de exactitud** con clases desbalanceadas: compárala siempre con el modelo «siempre no».
- **Olvidar `stratify=y`**: con 4 % de positivos, un corte al azar puede dejar la prueba casi sin días malos.
- **No escalar antes de K-Means**: la variable con números más grandes decide sola los grupos.
- **Creer que el cluster es «la verdad»**: es una agrupación útil, no una categoría oficial; cambia con *k* y con las variables.

## Cierre

1. **Clasificación** (supervisada): mismo `fit`/`predict`, pero la exactitud engaña con clases
   desbalanceadas → mira la **matriz de confusión**, la **precisión** y el **recall**.
2. **Qué error es más caro** es una decisión de política pública: `class_weight="balanced"` sube el recall a costa de la precisión.
3. **Clustering** (no supervisado): K-Means sobre perfiles **estandarizados** descubre grupos que tú interpretas.

> *En R2-07 aprenderás a evaluar con rigor: fuga de datos, validación cruzada y por qué con datos
> en el tiempo **no** se puede barajar.*

---
*Fuente de datos: Sistema de Información Nacional de Calidad del Aire (SINCA), Ministerio del Medio Ambiente.*